In [ ]:
# ============================================================
# Cell 1: Install dependencies
# ============================================================
# Run this cell first in Google Colab
!pip install transformers sentencepiece accelerate -q


In [ ]:
# ==========================================================
# 上传 csv 含sc150
# ============================================================
from google.colab import files
uploaded = files.upload()   



Saving sc150_human.csv to sc150_human.csv


In [ ]:
# ============================================================
# Cell 2: Load CSV and preview data
# ============================================================
import pandas as pd

# Load the source CSV file
INPUT_FILE  = "sc150_human.csv"
OUTPUT_FILE = "base_nllb_en150.csv"
COL_SRC     = "四川话原句"        # Column to translate
COL_TGT     = "nllb_bs"  # Output column name for translated text

df = pd.read_csv(INPUT_FILE)

# Sanity check: confirm the target column exists
assert COL_SRC in df.columns, (
    f"Column '{COL_SRC}' not found. Available columns: {list(df.columns)}"
)

print(f"Loaded {len(df)} rows from '{INPUT_FILE}'")
print(df[[COL_SRC]].head())



Loaded 999 rows from 'sc150_human.csv'
                                          四川话原句
0  所以说养成老习惯的看到老严来了他本人心中有个这个习惯甚至就是不是自己屋头的亲生妈老汉儿他
1               嫂子在扣你吗我一天是管不住他我喊大姐过过来制止他这摔摔摔老汉儿
2           其实我对你也没得啥子要求的就像你看到的那个像我老汉儿对我妈恁个好就行了
3                            你老汉儿不傻眼泪只是有点儿发披风哟西
4                         老汉儿天气越来冷了哈是农村比城头要低两对儿


In [ ]:
# ============================================================
# Cell 3: Load NLLB model and tokenizer
# ============================================================
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# ── Model options ─────────────────────────────────────────
# "facebook/nllb-200-distilled-600M"  → Lightest, fastest (recommended for free Colab)
# "facebook/nllb-200-distilled-1.3B"  → Better accuracy, runs on T4
# "facebook/nllb-200-1.3B"            → Non-distilled, highest accuracy
MODEL_NAME = "facebook/nllb-200-distilled-600M"

# NLLB language codes
SRC_LANG = "zho_Hans"   # Simplified Chinese (Mandarin baseline; covers Sichuan dialect text)
TGT_LANG = "eng_Latn"   # English

print(f"Loading model: {MODEL_NAME} ...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,   # Half precision to reduce VRAM usage
    device_map="auto",           # Automatically place on GPU if available
)

# Set model to inference mode (no gradient tracking needed)
model.eval()

print("Model loaded successfully.")



Loading model: facebook/nllb-200-distilled-600M ...


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded successfully.


In [ ]:
# ============================================================
# Cell 4: Define translation function
# ============================================================

def translate_nllb(texts: list[str], batch_size: int = 8) -> list[str]:
    """
    Translate a list of source texts to English using the NLLB model.

    Args:
        texts:      List of source-language strings to translate.
        batch_size: Number of sentences processed per forward pass.
                    Reduce if you hit CUDA out-of-memory errors.

    Returns:
        List of translated English strings (same order as input).
    """
    translations = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]

        # Tokenize the batch with the source language token
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256,        # Max input token length; increase for longer sentences
        ).to(model.device)

        # Force the decoder to start with the target language token
        forced_bos_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                forced_bos_token_id=forced_bos_token_id,
                max_new_tokens=256,    # Max output token length
                num_beams=4,           # Beam search width; higher = better quality, slower
                early_stopping=True,
            )

        # Decode output tokens back to strings, skipping special tokens
        batch_translations = tokenizer.batch_decode(
            output_ids, skip_special_tokens=True
        )
        translations.extend(batch_translations)

        # Progress indicator
        done = min(i + batch_size, len(texts))
        print(f"  Translated {done}/{len(texts)} sentences ...", end="\r")

    print()  # Newline after progress indicator
    return translations


In [ ]:
# ============================================================
# Cell 5: Run translation and save output CSV
# ============================================================

print("Starting NLLB translation ...")

source_texts = df[COL_SRC].astype(str).tolist()

# Translate all sentences
df[COL_TGT] = translate_nllb(source_texts, batch_size=8)

# Save the enriched DataFrame to a new CSV file
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")  # utf-8-sig for Excel compatibility

print(f"\nDone! Results saved to '{OUTPUT_FILE}'")
print(f"Total rows translated: {len(df)}")
print("\nSample output:")
print(df[[COL_SRC, COL_TGT]].head(5).to_string(index=False))


Starting NLLB translation ...
  Translated 999/999 sentences ...

Done! Results saved to 'base_nllb_en150.csv'
Total rows translated: 999

Sample output:
                                       四川话原句                                                                                                                                                    nllb_bs
所以说养成老习惯的看到老严来了他本人心中有个这个习惯甚至就是不是自己屋头的亲生妈老汉儿他 So to say that a person who is accustomed to growing old to see old age come into his own mind, this habit is not even the firstborn of his own household.
             嫂子在扣你吗我一天是管不住他我喊大姐过过来制止他这摔摔摔老汉儿                                                                      I couldn't handle him one day I yelled at my older sister to stop him from wrestling.
         其实我对你也没得啥子要求的就像你看到的那个像我老汉儿对我妈恁个好就行了                                                                          In fact, I didn't have to ask you to be as kind as the old man you saw did to me.
                          你老汉儿不傻眼泪只是有点儿发披风哟西  

In [ ]:
# ============================================================
# Cell 6: Load MBART model and tokenizer
# ============================================================
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# ── Model options ─────────────────────────────────────────
# "facebook/mbart-large-50-many-to-many-mmt"  → Supports 50 languages, recommended
# "facebook/mbart-large-cc25"                 → Older version, 25 languages only
MBART_MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"

# MBART-50 language codes (different format from NLLB)
MBART_SRC_LANG = "zh_CN"   # Simplified Chinese
MBART_TGT_LANG = "en_XX"   # English

# Reuse float16 dtype to save VRAM (same as NLLB setup)
DTYPE = torch.float16

print(f"Loading MBART model: {MBART_MODEL_NAME} ...")

# Step 1: load tokenizer first — tgt_lang_id depends on it
mbart_tokenizer = AutoTokenizer.from_pretrained(MBART_MODEL_NAME)
mbart_tokenizer.src_lang = MBART_SRC_LANG
mbart_tokenizer.tgt_lang = MBART_TGT_LANG

# Step 2: resolve target language token ID after tokenizer is ready
# convert_tokens_to_ids() is safer than lang_code_to_id[] dict lookup
# because it raises a clear error if the language code is not in vocabulary
tgt_lang_id = mbart_tokenizer.convert_tokens_to_ids(MBART_TGT_LANG)
print(f"Target language '{MBART_TGT_LANG}' resolved to token ID: {tgt_lang_id}")

# Step 3: load model after tgt_lang_id is confirmed
mbart_model = AutoModelForSeq2SeqLM.from_pretrained(
    MBART_MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto",         # Automatically place on GPU if available
)

# Step 4: set generation parameters on generation_config (NOT model.config)
# Transformers ≥4.40 raises ValueError if model.config is used for generation params
mbart_model.generation_config.forced_bos_token_id    = tgt_lang_id
mbart_model.generation_config.decoder_start_token_id = tgt_lang_id

# Step 5: set model to inference mode (no gradient tracking needed)
mbart_model.eval()

print("MBART model loaded successfully.")


Loading MBART model: facebook/mbart-large-50-many-to-many-mmt ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

Target language 'en_XX' resolved to token ID: 250004


model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

MBART model loaded successfully.


In [ ]:
# ============================================================
# Cell 7: Define MBART translation function
# ============================================================

def translate_mbart(texts: list[str], batch_size: int = 8) -> list[str]:
    """
    Translate a list of source texts to English using the MBART-50 model.

    Args:
        texts:      List of source-language strings to translate.
        batch_size: Number of sentences processed per forward pass.
                    Reduce to 4 or 2 if you hit CUDA out-of-memory errors.

    Returns:
        List of translated English strings (same order as input).
    """
    translations = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]

        # Tokenize with source language context already set on the tokenizer
        inputs = mbart_tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256,        # Max input token length; increase for longer sentences
        ).to(mbart_model.device)

        with torch.no_grad():
            # forced_bos_token_id is NOT passed here — it is already registered
            # on mbart_model.generation_config in Cell 6, which generate() reads
            # automatically. Passing it again via model.config caused the ValueError.
            # update the default hyperparameters for generate, to avoid repetition loop etc.
            output_ids = mbart_model.generate(
                **inputs,

                # language control
                forced_bos_token_id=tgt_lang_id,

                # safer decoding
                do_sample=True,
                top_p=0.9,
                temperature=0.7,

                # anti repetition
                repetition_penalty=2.0,
                no_repeat_ngram_size=3,

                # IMPORTANT
                max_new_tokens=40,
                min_new_tokens=3,

                early_stopping=True,
            )
        # Decode output token IDs back to strings, skipping special tokens
        batch_translations = mbart_tokenizer.batch_decode(
            output_ids, skip_special_tokens=True
        )
        translations.extend(batch_translations)

        # Progress indicator
        done = min(i + batch_size, len(texts))
        print(f"  Translated {done}/{len(texts)} sentences ...", end="\r")

    print()  # Newline after progress indicator
    return translations



In [ ]:
# ============================================================
# Cell 8: Run MBART translation and save output CSV
# ============================================================

MBART_OUTPUT_FILE  = "base_mbart_en150.csv"
MBART_COL_TGT      = "mbart_translation"   # New column name for MBART results

print("Starting MBART translation ...")
source_texts = df[COL_SRC].astype(str).tolist()
# Reuse the same source texts already loaded from sc150_human.csv (Cell 2)
df[MBART_COL_TGT] = translate_mbart(source_texts, batch_size=8)

# Save to a separate CSV file, independent of the NLLB output
df[[COL_SRC, MBART_COL_TGT]].to_csv(
    MBART_OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig",      # utf-8-sig ensures correct display in Excel
)

print(f"\nDone! Results saved to '{MBART_OUTPUT_FILE}'")
print(f"Total rows translated: {len(df)}")
print("\nSample output:")
print(df[[COL_SRC, MBART_COL_TGT]].head(5).to_string(index=False))


Starting MBART translation ...


[transformers] Both `max_new_tokens` (=40) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=40) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=40) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=40) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=40) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=40) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=40) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=40) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=40) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=40) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=40) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=40) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Translated 999/999 sentences ...

Done! Results saved to 'base_mbart_en150.csv'
Total rows translated: 999

Sample output:
                                       四川话原句                                                                                                              mbart_translation
所以说养成老习惯的看到老严来了他本人心中有个这个习惯甚至就是不是自己屋头的亲生妈老汉儿他                                        He had a habit in his own heart, even if it wasn't his mother-in-law who lived at home.
             嫂子在扣你吗我一天是管不住他我喊大姐过过来制止他这摔摔摔老汉儿 My sister is holding you I can't handle him one day I called my older sister over to stop him from falling and falling old man
         其实我对你也没得啥子要求的就像你看到的那个像我老汉儿对我妈恁个好就行了                                    In fact, I don't have to ask you anything. Just like the old man you saw is good for my mom
                          你老汉儿不傻眼泪只是有点儿发披风哟西                                                                      You old man are not silly tears just a little bit of hair
               